# Clock Setup

## Overview
This document describes the procedures and methods to setup a clock sequencer which is used to trigger a pulse sequence at a fixed frequency or event cycle. Clock sequencers are the backbone of measurement aquisition systems since they determine how often and when measurements are taken. As such, it is important to understand all the different configurations for clock sequencers, especially in the case for OpenSync.

## Purpose
To help users reach an optimal clock sequencer config, a set of examples and explanations are given for each major configuration mode. Since clock sequencer parameters are the main topic of this discussion, please note that the loading and executing of clock configs will not be discussed in this document.

## Creating the Clock Config
A basic clock config can be obtained from calling the function `get_clock_params` from the opensync namespace. All clock parameters are contained within this dictionary. It is typically not recommended to modify the dictionary of clock parameters and instead one should prioritize using helepr functions which check for invalid inputs. 

In [1]:
from opensync import opensync

In [2]:
clock_config = opensync.get_clock_params()
clock_config

{'clock_id': 0,
 'clock_res': 'high_res',
 'clock_mode': 'internal',
 'clock_units': 'hz',
 'clock_freq': [1.0],
 'clock_iter': [10],
 'trigger_id': 0,
 'trigger_mode': 'immediate',
 'trigger_edge': 'positive',
 'trigger_level': 'high',
 'trigger_units': 'us',
 'trigger_skips': 0,
 'trigger_delay': 0.0,
 'trigger_count': 10}

As you can see, there is a lot of different ways a clock sequencer can be configured. As a start, we will go over the first few basic parameters which do not change the way the clock sequencer runs. 

The `clock_id` represents the clock sequencer channel we want to modify. Each OpenSync device has three independent clock channels that we can modify and enable. The default resolution of a clock channel is `high_res` and has a resolution of four (4) nanoseconds. For very low repetition frequencies, the clock resolution can be decreased so the internal clock counters do not overflow and issue an error. When setting the repetition frequence of a clock through `clock_freq`, it is important to also set the `clock_units` which are hz (hertz), khz (kilohertz), and mhz (megahertz). This prevents one from accidentally executing a pulse sequence too fast or too slow for apparently no reason. Finally, a clock that is internally triggered (more on that later) can be repeated `clock_iter` times which may dictate how many aquisitions a measurement system performs at the chosen frequency. Note that `clock_freq` and `clock_iter` are in a list. This is because multiple frequencies and iterations can be specified during an aquisition run.

The next few parameters are slightly mroe complicated. The mode of a clock sequencer can be changed by using `clock_mode`. There are currently two supported clock modes: `internal` and `external`. As the names suggest, an internal clock uses a software clock from within the synchronizer to send signals to a pulse sequencer. Contrarily, an external clock does not use an internal software clock and relies solely on an external trigger. The way a clock mode behaves is dictated by `trigger_mode`. This is where things may become tricky. A trigger mode that contradicts a clock mode (e.g., a clock_mode of internal and trigger_mode of edge) will cause the synchronizer to silently ignore the clock sequencer. One has to make sure what their clock and trigger modes are of sane decisions. As a visual aid, below is a diagram to visualize clock and trigger modes along with what they do.

 - clock_mode: internal
   - trigger_mode: immediate
     - Clock sequencer runs in internal software clock mode and does not accept external events.
   - trigger_mode: gate
     - trigger_level: high
       - Clock sequencer runs in internal software clock mode, but only signal a pulse sequence should occur when an external trigger level is high.
     - trigger_level: low
       - Clock sequencer runs in internal software clock mode, but only signal a pulse sequence should occur when an external trigger level is low.
   - trigger_mode: edge
     - trigger_edge: positive
       - Clock sequencer runs in internal software clock mode, but only starts once a positive slope from an external trigger is received.
     - trigger_edge: negative
       - Clock sequencer runs in internal software clock mode, but only starts once a negative slope from an external trigger is received.
 - clock_mode: external
   - trigger_mode: immediate
     - Invalid, an externally triggered clock cannot run immediately on its own.
   - trigger_mode: gate
     - Invalid, an externally triggered clock cannot check if the output level is high or low and emit pulse sequence signals based on that information.
   - trigger_mode: edge
     - trigger_edge: positive
       - Clock sequencer runs in external clock mode, but only emits a pulse sequence signal once a positive slope from an external trigger is received.
     - trigger_edge: negative
       - Clock sequencer runs in external clock mode, but only emits a pulse sequance signal once a negative slope from an external trigger is received.
      
The rest of the trigger parameters are rather basic. `trigger_id` selects the trigger channel where the clock sequencer will accept an event. `trigger_units` sets the units in microseconds, milliseconds, etc, for delays. `trigger_skip` specifies how many external trigger events to skip before sending a pulse sequence signal. `trigger_delay` adds an artificial delay between the accepted external trigger event and when a pulse sequence signal is sent. Finally, `trigger_counts` specifies the total number of aquisitions or repetitions an externally-controlled clock should perform. For more detailed documentation, refer to the OpenSync API which goes into specific detail on the functions which modify the aforementioned parameters.

## Example 1: Freerun Aquisition at 1 kHz for 1,000 Iterations
For a simple particle image velocimetry (PIV) experiment, many do not use external triggers and simply want to capture a set of image pairs. In this example, 1,000 pulse sequence signals are sent at a frequency of one (1) kilohertz. With a suitable pulse config, this would result in 1,000 image pairs of high temporal resolution. The entire aquisition time would be around a single second.

In [3]:
# Get default clock config
clock_config = opensync.get_clock_params()

In [4]:
# Set clock channel to 0
clock_config = opensync.config_clock_id(
    clock_config,
    channel_id=0
)

In [5]:
# Make sure to set the clock units to kHz or use 1,000 Hz
clock_config = opensync.config_clock_units(
    clock_config,
    units='khz'
)

In [6]:
# Then change the clock frequency to 1 kHz
clock_config = opensync.config_clock_freq(
    clock_config,
    freq=1.0
)

In [7]:
# Then change the total amount of clock reps to 1,000
clock_config = opensync.config_clock_iter(
    clock_config,
    iterations=1000
)

In [8]:
# Finally, make sure to set the trigger mode to immediate and clock mode to internal
clock_config = opensync.config_clock_mode(
    clock_config,
    clock_mode='internal'
)

clock_config = opensync.config_clock_trigger_mode(
    clock_config,
    trigger_mode='immediate'
)

## Example 2: Freerun Aquisition at 1 kHz for 1,000 Iterations Only When Gate is High
For a slighly more complicated experiment, there may be times where one does not want to aquire images while the internal clock is still running. For instance, lets say there is an experiment where a submersible is moving upstream in a water channel. When the submersible is not in the field of view of the measurement system, it may be a good idea to not acquire images in order to save memeory space. As such, an external trigger gated to the internal clock can be used to disable and enable pulse sequencer fire signals. This allows the acquisition system to only capture images/measurements when the submersible is in view and otherwise 

Expanding upon the previous example, we will be setting `trigger_mode` to gated and `trigger_level` to `high`. Of course we may want a different acquisition frequence since one kilohertz is a little fast, but that is outside the scope of this example.

In [9]:
# Set trigger channel to 0
clock_config = opensync.config_clock_trigger_id(
    clock_config,
    channel_id=0
)

In [10]:
# Set trigger mode to gated
clock_config = opensync.config_clock_trigger_mode(
    clock_config,
    trigger_mode='gate'
)

In [11]:
# Now set trigger gate level. When set to high, clock signals are only sent when the trigger is high
clock_config = opensync.config_clock_trigger_level(
    clock_config,
    level='high'
)

## Example 3: Externally Triggered Aquisition with 9 Trigger Skips and for 1,000 Iterations
Some PIV experiments rely on external events in order to synchronize what is intended to be measured with the measurement system. For instance, a simple PIV experiment measuring the phase average flow of a computer fan would require an external trigger event to make sure the measurement system acquires images at the same exact fan blade orientation each time. Assuming the external trigger event mechanism is worked out, the trigger frequency may be simply too high for the current PIV system. As such, some external triggers have to be skipped in order to not exceed the specifications of the PIV system. Let's say that the fan spins at 500 revolutions per minute (RPM). An external trigger even would occur every 20 milliseconds, or an aquisition rate of 500 Hz. If our PIV system can only aquire images at something along the lines of 50 double pulsed images a second (e.g., 100 frames per second), then nine (9) trigger signal skips would be required in order to get the aquisition rate down to 50 Hz. Additionally, lets say we want to measure at a slight offset from when the accepted trigger occurs due to the fan blade orientation. A delay can be artificially be added to make sure everything is in proper alignment during measurement acquisition.

In [12]:
# Get default clock config
clock_config = opensync.get_clock_params()

In [13]:
# Set clock channel to 0
clock_config = opensync.config_clock_id(
    clock_config,
    channel_id=0
)

In [14]:
# Set clock mode to external
clock_config = opensync.config_clock_mode(
    clock_config,
    clock_mode='external'
)

Note that when a clock mode is set to `external`, no other clock parameters are used. This is because we are no longer using an internal software clock. Thus, we now configure the clock trigger which drives the whole acquisition frequency.

In [15]:
# Set trigger channel to 0
clock_config = opensync.config_clock_trigger_id(
    clock_config,
    channel_id=0
)

In [16]:
# Set trigger mode to edge
clock_config = opensync.config_clock_trigger_mode(
    clock_config,
    trigger_mode='edge'
)

In [17]:
# Now set the trigger edge. For rising edge (e.g., from low to high), we want to set the edge to positive
clock_config = opensync.config_clock_trigger_edge(
    clock_config,
    edge='positive'
)

In [18]:
# Set trigger units to microseconds to make sure we add the correct delay scale
clock_config = opensync.config_clock_trigger_units(
    clock_config,
    units='us'
)

In [19]:
# Now set the amount of trigger signals to skip. Sicne we want the tenth signal, set skip to 9
clock_config = opensync.config_clock_trigger_skips(
    clock_config,
    skips=9
)

In [20]:
# Now, set trigger delay to soemthing like 250 microseconds if that is the phase offset we want
clock_config = opensync.config_clock_trigger_delay(
    clock_config,
    delay=250.0
)

In [21]:
# Finally, set amount of acquisitions we want to perform
clock_config = opensync.config_clock_trigger_count(
    clock_config,
    count=1000
)